# HR Assistant Agent — Ground Truth Evaluations (boto3-direct)

This notebook demonstrates evaluation of an agentic application with ground truth using
[**Amazon Bedrock AgentCore Evaluations**](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)
using **direct boto3 API calls** throughout.

Unlike a higher-level wrapper approach, this variant avoids any convenience layers so
every AWS API call is visible and can be copied verbatim into customer pipelines.

| What a wrapper-based sample would use | What this notebook uses instead |
|---|---|
| Starter-toolkit deployment | `bedrock-agentcore-control.create_agent_runtime` / `update_agent_runtime` with direct `docker build` + `docker push` |
| SDK wrapper for per-session eval | `bedrock-agentcore.evaluate` (direct boto3) |
| SDK wrapper for dataset-driven eval | Python loop over the ground-truth dataset, invoking the agent via `bedrock-agentcore.invoke_agent_runtime` and evaluating with `evaluate` / `start_batch_evaluation` |

### The HR Assistant Agent

We deploy an **HR Assistant** for Acme Corp — a [Strands](https://strandsagents.com/) agent that helps employees with:
- PTO balance checks and time-off requests
- HR policy lookups (PTO, remote work, parental leave)
- Benefits information (health, dental, vision, 401k)
- Pay stub retrieval

### What You'll Learn
- How to deploy an agent to AgentCore Runtime using only `boto3` and the Docker CLI
- How to run per-session ad-hoc evaluations with `bedrock-agentcore.evaluate`
- How to loop over a ground-truth dataset, invoke the agent for each scenario, and evaluate the resulting CloudWatch sessions
- How to kick off a **batch evaluation** with `start_batch_evaluation`, poll `get_batch_evaluation` to a terminal state, and read back per-evaluator summaries
- How to interpret evaluation results across built-in evaluators (`Builtin.Correctness`, `Builtin.GoalSuccessRate`, `Builtin.TrajectoryExactOrderMatch`)

### Tutorial Details

| Information | Details |
|---|---|
| Agent framework | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation data plane | `bedrock-agentcore` (direct boto3 — no SDK wrappers) |
| Control plane | `bedrock-agentcore-control` |
| AWS services | AgentCore Runtime, AgentCore Evaluations, CloudWatch Logs, ECR |

### Prerequisites
- Python 3.10+
- `boto3 >= 1.43.0` (the release that merged the Evaluations data-plane operations onto the `bedrock-agentcore` client)
- AWS credentials available to the default boto3 session
- Docker running locally / in the kernel host (for agent container image build)
- A `Dockerfile` for the agent container co-located with `hr_assistant_agent.py`

## Step 1: Install Dependencies

Install only what we need for boto3-direct deployment and evaluation. We
**deliberately do not** pull in the SDK evaluation wrappers because this
notebook calls the data-plane API directly.

In [ ]:
!pip install -q 'boto3>=1.43.0' strands-agents

## Step 2: Configuration

Import libraries and configure your boto3 session. Edit `REGION` below if you
plan to deploy the agent into a different region.

We create only **two** clients:

| Client | Variable | Operations used |
|---|---|---|
| `bedrock-agentcore-control` | `cp` | `create_agent_runtime`, `update_agent_runtime`, `get_agent_runtime`, `list_agent_runtimes`, `create_evaluator` |
| `bedrock-agentcore` | `bac` | `invoke_agent_runtime`, `evaluate`, `start_batch_evaluation`, `get_batch_evaluation` |

In `boto3 >= 1.43.0`, the evaluation data-plane operations live on the same
`bedrock-agentcore` client as the runtime invoke API — no separate data-plane
client is needed.

In [ ]:
import json
import time
import uuid
from datetime import timedelta
from pathlib import Path

import boto3
from boto3.session import Session
from IPython.display import display, Markdown

# Edit this if you are deploying in a different region.
REGION = "us-west-2"

boto_session = Session(region_name=REGION)

# Control plane: manages agent runtimes + custom evaluators.
cp = boto3.client("bedrock-agentcore-control", region_name=REGION)

# Data plane: runtime invoke + evaluation operations (evaluate, start_batch_evaluation,
# get_batch_evaluation) all live on this single client in boto3 >= 1.43.0.
bac = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"Region : {REGION}")
print(f"cp     : {cp.meta.service_model.service_name}")
print(f"bac    : {bac.meta.service_model.service_name}")

## Step 3: Write the Agent Source

Write the HR assistant source to disk. The agent itself is preserved verbatim
from the upstream sample — only the **deployment** path differs in this
notebook.

In [ ]:
%%writefile hr_assistant_agent.py
"""
HR Assistant Agent — Strands agent deployed on Bedrock AgentCore Runtime.

Tools (deterministic / mock data for reproducible evaluations):
  get_pto_balance        — remaining PTO days for an employee
  submit_pto_request     — request time off
  lookup_hr_policy       — company policy documents
  get_benefits_summary   — health, dental, vision, 401k, life insurance details
  get_pay_stub           — pay stub for a given period
"""

import logging
import re

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

# ---------------------------------------------------------------------------
# Mock data
# ---------------------------------------------------------------------------

_PTO_BALANCES = {
    "EMP-001": {"total_days": 15, "used_days": 5, "remaining_days": 10},
    "EMP-002": {"total_days": 15, "used_days": 12, "remaining_days": 3},
    "EMP-042": {"total_days": 20, "used_days": 7, "remaining_days": 13},
}

_HR_POLICIES = {
    "pto": (
        "PTO Policy: Full-time employees accrue 15 days of PTO per year (20 days after 3 years). "
        "PTO requests must be submitted at least 2 business days in advance. "
        "Unused PTO up to 5 days rolls over to the next year. "
        "PTO cannot be taken in advance of accrual."
    ),
    "remote_work": (
        "Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. "
        "Core collaboration hours are 10am-3pm local time. "
        "A dedicated workspace with reliable internet (25 Mbps+) is required. "
        "Employees must be reachable via Slack and email during core hours."
    ),
    "parental_leave": (
        "Parental Leave Policy: Primary caregivers receive 16 weeks of fully paid parental leave. "
        "Secondary caregivers receive 6 weeks of fully paid parental leave. "
        "Leave may begin up to 2 weeks before the expected birth or adoption date. "
        "Benefits continue unchanged during parental leave."
    ),
    "code_of_conduct": (
        "Code of Conduct: All employees are expected to treat colleagues, customers, and partners "
        "with respect and professionalism. Harassment, discrimination, and retaliation of any kind "
        "are strictly prohibited. Violations should be reported to HR or via the anonymous hotline."
    ),
}

_BENEFITS = {
    "health": (
        "Health Insurance: The company covers 90% of premiums for employee-only coverage and 75% "
        "for family coverage. Plans available: Blue Shield PPO, Kaiser HMO, and HDHP with HSA. "
        "Annual deductible: $500 (PPO), $0 (HMO), $1,500 (HDHP). "
        "Open enrollment is each November for the following calendar year."
    ),
    "dental": (
        "Dental Insurance: 100% coverage for preventive care (cleanings, X-rays). "
        "80% coverage for basic restorative care (fillings, extractions). "
        "50% coverage for major restorative care (crowns, bridges). "
        "Annual maximum benefit: $2,000 per person. Orthodontia lifetime maximum: $1,500."
    ),
    "vision": (
        "Vision Insurance: Annual eye exam covered in full. "
        "Frames or contacts allowance: $200 per year. "
        "Laser vision correction discount: 15% off at participating providers."
    ),
    "401k": (
        "401(k) Plan: The company matches 100% of employee contributions up to 4% of salary. "
        "An additional 50% match on the next 2% (total effective match up to 5%). "
        "Employees are eligible to contribute immediately; company match vests over 3 years. "
        "2026 IRS contribution limit: $23,500 (under 50), $31,000 (age 50+)."
    ),
    "life_insurance": (
        "Life Insurance: Basic life insurance of 2x annual salary provided at no cost. "
        "Employees may purchase supplemental coverage up to 5x salary during open enrollment. "
        "Accidental death and dismemberment (AD&D) coverage equal to basic life benefit is included."
    ),
}

_PAY_STUBS = {
    ("EMP-001", "2025-12"): {
        "gross_pay": 8333.33,
        "federal_tax": 1458.33,
        "state_tax": 416.67,
        "social_security": 516.67,
        "medicare": 120.83,
        "health_premium": 125.00,
        "401k_contribution": 333.33,
        "net_pay": 5362.50,
        "period": "December 2025",
    },
    ("EMP-001", "2026-01"): {
        "gross_pay": 8333.33,
        "federal_tax": 1458.33,
        "state_tax": 416.67,
        "social_security": 516.67,
        "medicare": 120.83,
        "health_premium": 125.00,
        "401k_contribution": 333.33,
        "net_pay": 5362.50,
        "period": "January 2026",
    },
    ("EMP-042", "2026-01"): {
        "gross_pay": 10416.67,
        "federal_tax": 1875.00,
        "state_tax": 520.83,
        "social_security": 645.83,
        "medicare": 151.04,
        "health_premium": 200.00,
        "401k_contribution": 416.67,
        "net_pay": 6607.30,
        "period": "January 2026",
    },
}

_PTO_REQUEST_COUNTER = {"n": 0}


# ---------------------------------------------------------------------------
# Strands tools
# ---------------------------------------------------------------------------


@tool
def get_pto_balance(employee_id: str) -> dict:
    """
    Return the current PTO balance for an employee.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)

    Returns:
        Dict with total_days, used_days, and remaining_days.
    """
    balance = _PTO_BALANCES.get(employee_id)
    if balance:
        return {"employee_id": employee_id, **balance}
    return {"employee_id": employee_id, "error": f"Employee {employee_id} not found."}


@tool
def submit_pto_request(
    employee_id: str,
    start_date: str,
    end_date: str,
    reason: str = "Personal time off",
) -> dict:
    """
    Submit a PTO request for an employee.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)
        start_date:  First day of leave in YYYY-MM-DD format
        end_date:    Last day of leave in YYYY-MM-DD format
        reason:      Optional reason for the request

    Returns:
        Dict with request_id, status, and confirmation message.
    """
    _PTO_REQUEST_COUNTER["n"] += 1
    request_id = f"PTO-2026-{_PTO_REQUEST_COUNTER['n']:03d}"
    return {
        "request_id": request_id,
        "employee_id": employee_id,
        "start_date": start_date,
        "end_date": end_date,
        "reason": reason,
        "status": "APPROVED",
        "message": f"PTO request {request_id} approved for {employee_id} from {start_date} to {end_date}.",
    }


@tool
def lookup_hr_policy(topic: str) -> dict:
    """
    Look up a company HR policy document by topic.

    Args:
        topic: Policy topic. Supported values: pto, remote_work, parental_leave, code_of_conduct

    Returns:
        Dict with topic and policy_text.
    """
    key = topic.lower().replace(" ", "_").replace("-", "_")
    text = _HR_POLICIES.get(key)
    if text:
        return {"topic": topic, "policy_text": text}
    return {
        "topic": topic,
        "error": f"Policy '{topic}' not found. Available: {list(_HR_POLICIES.keys())}",
    }


@tool
def get_benefits_summary(benefit_type: str) -> dict:
    """
    Return a summary of a specific employee benefit.

    Args:
        benefit_type: Type of benefit. Supported values: health, dental, vision, 401k, life_insurance

    Returns:
        Dict with benefit_type and summary text.
    """
    key = benefit_type.lower().replace(" ", "_").replace("-", "_")
    text = _BENEFITS.get(key)
    if text:
        return {"benefit_type": benefit_type, "summary": text}
    return {
        "benefit_type": benefit_type,
        "error": f"Benefit '{benefit_type}' not found. Available: {list(_BENEFITS.keys())}",
    }


@tool
def get_pay_stub(employee_id: str, period: str) -> dict:
    """
    Retrieve a pay stub for an employee for a specific pay period.

    Args:
        employee_id: Employee identifier (e.g. EMP-001)
        period:      Pay period in YYYY-MM format (e.g. 2026-01)

    Returns:
        Dict with gross pay, deductions, and net pay.
    """
    stub = _PAY_STUBS.get((employee_id, period))
    if stub:
        return {"employee_id": employee_id, **stub}
    return {
        "employee_id": employee_id,
        "period": period,
        "error": f"Pay stub not found for {employee_id} period {period}.",
    }


# ---------------------------------------------------------------------------
# Agent
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a helpful HR Assistant for Acme Corp.

You help employees with:
- Checking PTO (paid time off) balances
- Submitting PTO requests
- Looking up HR policies (PTO, remote work, parental leave, code of conduct)
- Understanding employee benefits (health, dental, vision, 401k, life insurance)
- Retrieving pay stub information

Always use the available tools to answer questions accurately. Do not make up
policy details, benefit amounts, or pay information — look them up.
Be concise, professional, and friendly."""

_MODEL = BedrockModel(model_id="us.amazon.nova-lite-v1:0")
_TOOLS = [
    get_pto_balance,
    submit_pto_request,
    lookup_hr_policy,
    get_benefits_summary,
    get_pay_stub,
]

# Session cache: session_id -> Agent (preserves conversation history across turns)
_SESSION_AGENTS: dict[str, Agent] = {}


@app.entrypoint
async def invoke(payload, context):
    """Handle an agent invocation from AgentCore Runtime."""
    prompt = payload.get("prompt", "")
    session_id = context.session_id
    logger.info("Received prompt (session=%s): %s", session_id, prompt[:80])

    if session_id and session_id in _SESSION_AGENTS:
        agent = _SESSION_AGENTS[session_id]
    else:
        agent = Agent(model=_MODEL, tools=_TOOLS, system_prompt=SYSTEM_PROMPT)
        if session_id:
            _SESSION_AGENTS[session_id] = agent

    parts = []
    async for event in agent.stream_async(prompt):
        if "data" in event:
            parts.append(str(event["data"]))
    response = "".join(parts)
    # Strip inline <thinking>...</thinking> blocks so spans contain only the final answer
    response = re.sub(
        r"<thinking>.*?</thinking>", "", response, flags=re.DOTALL
    ).strip()
    return response


if __name__ == "__main__":
    app.run()

## Step 4: Deploy the HR Assistant Agent (boto3 + Docker CLI)

We deploy the HR Assistant to **AgentCore Runtime** using the AWS SDK
directly — no higher-level deployment toolkit involved. The steps are:

1. **Resolve account + ECR repo** — read `sts.get_caller_identity()` and ensure
   an ECR repository exists for the agent image.
2. **ECR authentication** — use `aws ecr get-login-password` piped into
   `docker login` to obtain a short-lived token.
3. **Docker build** — build a container image from `hr_assistant_agent.py`
   and a local `Dockerfile` sitting next to it.
4. **Docker push** — push the image to the ECR repository.
5. **CreateAgentRuntime / UpdateAgentRuntime** — register or update the
   agent endpoint via `bedrock-agentcore-control`. We check
   `list_agent_runtimes` first so re-running the cell performs an update
   instead of creating a duplicate runtime.

> **Note** — this cell requires Docker running locally and an IAM role the
> AgentCore service can assume (`AGENT_EXECUTION_ROLE_ARN`). Edit the
> placeholders below before running.

In [ ]:
# ---- Editable placeholders ---------------------------------------------------
AGENT_NAME = "hr_assistant_eval_tutorial"
ECR_REPOSITORY_NAME = "hr-assistant-eval-tutorial"
IMAGE_TAG = "latest"
# IAM role the AgentCore service assumes to run the agent. Must trust
# bedrock-agentcore.amazonaws.com and grant permissions for Bedrock model
# invocation + CloudWatch Logs. Replace with your account's role ARN.
AGENT_EXECUTION_ROLE_ARN = "arn:aws:iam::REPLACE_ACCOUNT_ID:role/AgentCoreAgentExecutionRole"

# ---- Additional boto3 clients ------------------------------------------------
ecr = boto3.client("ecr", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY_NAME}"
IMAGE_URI = f"{ECR_URI}:{IMAGE_TAG}"
print(f"ACCOUNT_ID : {ACCOUNT_ID}")
print(f"IMAGE_URI  : {IMAGE_URI}")

# ---- 1. Ensure ECR repository exists -----------------------------------------
try:
    ecr.describe_repositories(repositoryNames=[ECR_REPOSITORY_NAME])
    print(f"ECR repo {ECR_REPOSITORY_NAME!r} already exists.")
except ecr.exceptions.RepositoryNotFoundException:
    ecr.create_repository(repositoryName=ECR_REPOSITORY_NAME)
    print(f"Created ECR repo {ECR_REPOSITORY_NAME!r}.")

In [ ]:
# ---- 2. ECR login ------------------------------------------------------------
# `aws ecr get-login-password` returns a short-lived token; pipe it into
# `docker login` so Docker can push to the private ECR repo.
!aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com

In [ ]:
# ---- 3. Docker build ---------------------------------------------------------
# Assumes a Dockerfile co-located with hr_assistant_agent.py. A minimal
# Dockerfile looks like:
#
#   FROM public.ecr.aws/docker/library/python:3.11-slim
#   WORKDIR /app
#   COPY hr_assistant_agent.py ./
#   RUN pip install --no-cache-dir 'boto3>=1.43.0' bedrock-agentcore strands-agents
#   CMD ["python", "hr_assistant_agent.py"]
!docker build -t $IMAGE_URI .

In [ ]:
# ---- 4. Docker push ----------------------------------------------------------
!docker push $IMAGE_URI

In [ ]:
# ---- 5. CreateAgentRuntime / UpdateAgentRuntime ------------------------------
# Idempotency: if an agent runtime with this name already exists, update it
# in place rather than creating a duplicate.
existing_runtime_id = None
existing_runtime_arn = None

paginator = cp.get_paginator("list_agent_runtimes")
for page in paginator.paginate():
    for rt_summary in page.get("agentRuntimes", []):
        if rt_summary.get("agentRuntimeName") == AGENT_NAME:
            existing_runtime_id = rt_summary["agentRuntimeId"]
            existing_runtime_arn = rt_summary["agentRuntimeArn"]
            break
    if existing_runtime_id:
        break

runtime_args = {
    "agentRuntimeName": AGENT_NAME,
    "agentRuntimeArtifact": {
        "containerConfiguration": {"containerUri": IMAGE_URI},
    },
    "roleArn": AGENT_EXECUTION_ROLE_ARN,
    "networkConfiguration": {"networkMode": "PUBLIC"},
}

if existing_runtime_id:
    print(f"Updating existing agent runtime {existing_runtime_id} ...")
    resp = cp.update_agent_runtime(
        agentRuntimeId=existing_runtime_id,
        **runtime_args,
    )
    AGENT_ID = existing_runtime_id
    AGENT_ARN = existing_runtime_arn
else:
    print(f"Creating new agent runtime {AGENT_NAME!r} ...")
    resp = cp.create_agent_runtime(**runtime_args)
    AGENT_ID = resp["agentRuntimeId"]
    AGENT_ARN = resp["agentRuntimeArn"]

CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"
SERVICE_NAME = AGENT_NAME

print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")
print(f"SERVICE_NAME : {SERVICE_NAME}")

## Step 5: Wait for the Runtime to Reach READY

Poll `bedrock-agentcore-control.get_agent_runtime` until the runtime reports
a terminal status. This replaces any SDK toolkit's internal polling loop.

In [ ]:
POLL_INTERVAL = 15   # seconds between status checks
MAX_WAIT = 600       # 10-minute timeout

elapsed = 0
while elapsed < MAX_WAIT:
    status_resp = cp.get_agent_runtime(agentRuntimeId=AGENT_ID)
    status = status_resp.get("status", "UNKNOWN")
    print(f"  [{elapsed:>3}s] status = {status}")
    if status in ("READY", "ACTIVE"):
        print(f"\nAgent is {status}. Proceeding.")
        break
    if status in ("FAILED", "CREATE_FAILED", "UPDATE_FAILED"):
        raise RuntimeError(
            f"Agent deployment failed with status {status!r}. Details: {status_resp}"
        )
    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise TimeoutError(
        f"Agent did not reach READY status within {MAX_WAIT}s. "
        "Check the AgentCore console for details."
    )

## Step 6: Invoke the Agent to Generate Sessions

Before we can evaluate, we need agent sessions with CloudWatch spans. We
invoke the agent for several scenarios and collect their session IDs for use
with per-session `evaluate` and batch evaluation later in the notebook.

This section uses the `bedrock-agentcore` client's `invoke_agent_runtime`
API directly. In `boto3 >= 1.43.0`, the same client also exposes the
evaluation operations (`evaluate`, `start_batch_evaluation`,
`get_batch_evaluation`) that we use in later sections.

In [ ]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """Send a single prompt to the HR assistant and return its text response."""
    resp = bac.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = resp["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: "):]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


def run_session(turns: list[str], session_prefix: str) -> str:
    """Invoke a multi-turn session and return its session ID."""
    session_id = f"{session_prefix}-{uuid.uuid4()}"
    print(f"Session: {session_id}")
    for turn_input in turns:
        print(f"  > {turn_input[:70]}")
        response = invoke_agent(turn_input, session_id)
        print(f"  < {response[:100]}")
    return session_id

In [ ]:
# --- Single-turn sessions ---

print("=== Single-Turn Sessions ===")

session_pto_balance = run_session(
    ["What is the current PTO balance for employee EMP-001?"],
    "pto-balance-check"
)

session_submit_pto = run_session(
    ["Please submit a PTO request for employee EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation."],
    "submit-pto-request"
)

session_pay_stub = run_session(
    ["Can you pull up the January 2026 pay stub for employee EMP-001?"],
    "pay-stub-lookup"
)

print("\nSingle-turn sessions created.")

In [ ]:
# --- Multi-turn session: PTO planning ---

print("=== Multi-Turn Session: PTO Planning ===")

session_pto_planning = run_session(
    [
        "How many PTO days do I have left? My employee ID is EMP-001.",
        "Great. I'd like to take December 23 to December 25 off. Please submit a request.",
        "Remind me — what is the policy on rolling over unused PTO?",
    ],
    "pto-planning-session"
)

print("\nMulti-turn session created.")

In [ ]:
# --- Multi-turn session: New employee onboarding ---

print("=== Multi-Turn Session: New Employee Onboarding ===")

session_onboarding = run_session(
    [
        "I just joined the company. What is the remote work policy?",
        "How much PTO do I get as a new employee?",
        "What life insurance benefit does the company provide?",
        "Can you check the current PTO balance for employee EMP-042?",
    ],
    "new-employee-onboarding"
)

print("\nAll sessions created. Waiting 60s for CloudWatch log ingestion...")
time.sleep(60)
print("Ready to evaluate.")

## Step 7: Create Custom (LLM-as-a-Judge) Evaluators

In addition to built-in evaluators, we can define our own evaluation criteria
using LLM-as-a-Judge custom evaluators. These accept natural language
instructions that reference **ground truth placeholders** automatically
substituted at evaluation time.

### Ground truth placeholders

| Level | Available placeholders |
|---|---|
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` |
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` |

We create two custom evaluators — one at each level — and use them alongside
built-ins in later sections.

In [ ]:
_SUFFIX = uuid.uuid4().hex[:8]

# ---------------------------------------------------------------------------
# Trace-level: HRResponseSimilarity
# Compares the agent's response to the expected_response reference input.
# {assistant_turn} → actual agent output
# {expected_response} → expectedResponse field in ReferenceInputs
# ---------------------------------------------------------------------------
print("Creating HRResponseSimilarity (TRACE) ...")
_resp_sim = cp.create_evaluator(
    evaluatorName=f"HRResponseSimilarity_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Compare the agent's response with the expected response.\n"
                "Agent response: {assistant_turn}\n"
                "Expected response: {expected_response}\n\n"
                "Rate how closely the agent's response matches the expected response. "
                "Focus on whether the key facts, numbers, and conclusions agree."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "not_similar",
                        "definition": "Response is factually different or missing key information from the expected response.",
                    },
                    {
                        "value": 0.5,
                        "label": "partially_similar",
                        "definition": "Response captures some expected content but omits or misrepresents parts.",
                    },
                    {
                        "value": 1.0,
                        "label": "highly_similar",
                        "definition": "Response is semantically equivalent to the expected response — all key facts match.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_RESPONSE_SIMILARITY_ID = _resp_sim["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_RESPONSE_SIMILARITY_ID}")

# ---------------------------------------------------------------------------
# Session-level: HRAssertionChecker
# Evaluates tool trajectory compliance and assertion satisfaction.
# {actual_tool_trajectory}   → tools the agent actually called
# {expected_tool_trajectory} → expectedTrajectory from ReferenceInputs
# {assertions}               → assertions list from ReferenceInputs
# ---------------------------------------------------------------------------
print("\nCreating HRAssertionChecker (SESSION) ...")
_assert_chk = cp.create_evaluator(
    evaluatorName=f"HRAssertionChecker_{_SUFFIX}",
    level="SESSION",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Evaluate whether the agent fulfilled the session requirements.\n\n"
                "Expected tool trajectory: {expected_tool_trajectory}\n"
                "Actual tool trajectory: {actual_tool_trajectory}\n"
                "Assertions to verify: {assertions}\n\n"
                "Score the agent on how well it followed the expected tool trajectory "
                "and satisfied every listed assertion."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "failed",
                        "definition": "Agent did not follow the trajectory and failed most assertions.",
                    },
                    {
                        "value": 0.5,
                        "label": "partial",
                        "definition": "Agent partially followed the trajectory or satisfied only some assertions.",
                    },
                    {
                        "value": 1.0,
                        "label": "passed",
                        "definition": "Agent followed the expected trajectory and satisfied all assertions.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_ASSERTION_CHECKER_ID = _assert_chk["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_ASSERTION_CHECKER_ID}")

print("\nCustom evaluators ready:")
print(f"  HRResponseSimilarity (TRACE)   : {CUSTOM_RESPONSE_SIMILARITY_ID}")
print(f"  HRAssertionChecker   (SESSION) : {CUSTOM_ASSERTION_CHECKER_ID}")

## Step 8: Per-Session Ad-Hoc Evaluation

This section replaces any higher-level per-session SDK wrapper with direct
boto3 calls on the `bedrock-agentcore` client.

### Note on the API name

The SDK JSON model defines the per-session operation as `Evaluate` (not
`StartEvaluation`). Boto3 translates this to `bac.evaluate(...)`. The request
shape mirrors what an SDK wrapper would send under the hood:

```python
bac.evaluate(
    evaluatorId="Builtin.Correctness",
    evaluationInput={"sessionSpans": [{"sessionId": "...", "logGroupNames": [...]}]},
    evaluationTarget={"sessionId": "..."},
    evaluationReferenceInputs=[{"expectedResponse": "..."}],
)
```

We loop over `(session_id, evaluator_id)` pairs and print each result.

In [ ]:
# Per-session reference inputs — ground-truth expectations per session.
PER_SESSION_REFERENCES = {
    session_pto_balance: {
        "expected_response": "Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
        "expected_trajectory": ["get_pto_balance"],
        "assertions": [
            "Agent called get_pto_balance with employee_id=EMP-001",
            "Agent reported 10 remaining PTO days",
        ],
    },
    session_submit_pto: {
        "expected_response": "PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
        "expected_trajectory": ["submit_pto_request"],
        "assertions": [
            "Agent called submit_pto_request for employee EMP-001",
            "Agent confirmed the PTO request was approved",
            "Agent provided a request ID (e.g. PTO-2026-001)",
        ],
    },
    session_pay_stub: {
        "expected_response": "EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
        "expected_trajectory": ["get_pay_stub"],
        "assertions": [
            "Agent called get_pay_stub for EMP-001 period 2026-01",
            "Agent reported the correct gross pay of $8,333.33",
            "Agent reported the correct net pay of $5,362.50",
        ],
    },
    session_pto_planning: {
        "expected_response": None,  # multi-turn — rely on trajectory + assertions
        "expected_trajectory": ["get_pto_balance", "submit_pto_request", "lookup_hr_policy"],
        "assertions": [
            "Agent correctly reported 10 remaining PTO days for EMP-001 in turn 1",
            "Agent submitted a PTO request for December 23-25, 2026 in turn 2",
            "Agent correctly stated the 5-day PTO rollover limit in turn 3",
        ],
    },
}

# (session_id, evaluator_id) pairs to evaluate.
# Built-in evaluators are routed by the service to the correct level (TRACE vs SESSION).
AD_HOC_PAIRS = [
    (session_pto_balance, "Builtin.Correctness"),
    (session_pto_balance, "Builtin.Helpfulness"),
    (session_pto_balance, CUSTOM_RESPONSE_SIMILARITY_ID),
    (session_submit_pto, "Builtin.Correctness"),
    (session_submit_pto, "Builtin.GoalSuccessRate"),
    (session_submit_pto, "Builtin.TrajectoryExactOrderMatch"),
    (session_pay_stub, "Builtin.Correctness"),
    (session_pay_stub, "Builtin.GoalSuccessRate"),
    (session_pto_planning, "Builtin.GoalSuccessRate"),
    (session_pto_planning, "Builtin.TrajectoryExactOrderMatch"),
    (session_pto_planning, CUSTOM_ASSERTION_CHECKER_ID),
]


def build_reference_inputs(session_id: str) -> list[dict]:
    """Shape the per-session ground truth into the evaluationReferenceInputs payload."""
    refs = PER_SESSION_REFERENCES.get(session_id, {})
    payload: dict = {}
    if refs.get("expected_response"):
        payload["expectedResponse"] = refs["expected_response"]
    if refs.get("expected_trajectory"):
        payload["expectedTrajectory"] = {"toolNames": refs["expected_trajectory"]}
    if refs.get("assertions"):
        payload["assertions"] = [{"text": a} for a in refs["assertions"]]
    return [payload] if payload else []


results_rows = []
for session_id, evaluator_id in AD_HOC_PAIRS:
    print(f"Evaluating session={session_id[:36]}... evaluator={evaluator_id}")
    try:
        resp = bac.evaluate(
            evaluatorId=evaluator_id,
            evaluationInput={
                "sessionSpans": [
                    {
                        "sessionId": session_id,
                        "logGroupNames": ["aws/spans", CW_LOG_GROUP],
                    }
                ]
            },
            evaluationTarget={"sessionId": session_id},
            evaluationReferenceInputs=build_reference_inputs(session_id),
        )
        results_rows.append({
            "session_id": session_id,
            "evaluator_id": evaluator_id,
            "response": resp,
        })
    except Exception as exc:  # noqa: BLE001 — surface per-row errors, don't abort the loop
        print(f"  ERROR: {exc}")
        results_rows.append({
            "session_id": session_id,
            "evaluator_id": evaluator_id,
            "error": str(exc),
        })

print(f"\nCompleted {len(results_rows)} ad-hoc evaluation calls.")

In [ ]:
# Pretty-print the ad-hoc evaluation results as a markdown table.
rows = ["| Session | Evaluator | Value | Label | Explanation |",
        "|---|---|---|---|---|"]
for row in results_rows:
    if "error" in row:
        rows.append(
            f"| `{row['session_id'][:20]}...` | `{row['evaluator_id'][:35]}` | ERR | — | {row['error'][:100]} |"
        )
        continue
    resp = row["response"] or {}
    # The Evaluate response shape contains evaluationResult with one or more results.
    for res in resp.get("evaluationResult", {}).get("results", []) or [resp]:
        value = str(res.get("value", res.get("score", "N/A")))
        lbl = str(res.get("label", res.get("rating", "")))
        explanation = (res.get("explanation", res.get("reason", "")) or "")[:120].replace("\n", " ")
        err_code = res.get("errorCode")
        if err_code:
            lbl = f"ERR:{err_code}"
            explanation = (res.get("errorMessage", "") or "")[:120]
        rows.append(
            f"| `{row['session_id'][:20]}...` | `{row['evaluator_id'][:35]}` | {value} | {lbl} | {explanation} |"
        )

display(Markdown("### Per-Session Ad-Hoc Evaluation Results\n\n" + "\n".join(rows)))

## Step 9: Dataset-Driven Evaluation

This section replaces any dataset-runner SDK wrapper with an explicit Python
loop. The structure is deliberately transparent:

```
For each scenario in the dataset:
    1. Create a fresh session_id (per-scenario).
    2. Invoke the agent for each turn via `bedrock-agentcore.invoke_agent_runtime`
       to populate CloudWatch spans.
    3. Record the session_id alongside the ground-truth metadata.
Wait for CloudWatch ingestion.
Evaluate — either per session via `bac.evaluate(...)` or by feeding the
collected session IDs to a batch run (see the next section).
```

We reuse the same five ground-truth scenario IDs as the earlier session walk:
`pto-balance-check`, `submit-pto-request`, `pay-stub-lookup`,
`pto-planning-session`, and `new-employee-onboarding`.

In [ ]:
# Inline dataset: scenario_id, turns, expected_trajectory, assertions, expected_response.
DATASET = [
    {
        "scenario_id": "pto-balance-check",
        "turns": [
            "What is the current PTO balance for employee EMP-001?",
        ],
        "expected_trajectory": ["get_pto_balance"],
        "assertions": [
            "Agent called get_pto_balance with employee_id=EMP-001",
            "Agent reported 10 remaining PTO days",
        ],
        "expected_response": "Employee EMP-001 has 10 remaining PTO days out of 15 total (5 days used).",
    },
    {
        "scenario_id": "submit-pto-request",
        "turns": [
            "Please submit a PTO request for employee EMP-001 from 2026-04-14 to 2026-04-16 for a family vacation.",
        ],
        "expected_trajectory": ["submit_pto_request"],
        "assertions": [
            "Agent called submit_pto_request for employee EMP-001",
            "Agent confirmed the PTO request was approved",
            "Agent provided a request ID (e.g. PTO-2026-001)",
        ],
        "expected_response": "PTO request submitted and approved for EMP-001 from 2026-04-14 to 2026-04-16.",
    },
    {
        "scenario_id": "pay-stub-lookup",
        "turns": [
            "Can you pull up the January 2026 pay stub for employee EMP-001?",
        ],
        "expected_trajectory": ["get_pay_stub"],
        "assertions": [
            "Agent called get_pay_stub for EMP-001 period 2026-01",
            "Agent reported the correct gross pay of $8,333.33",
            "Agent reported the correct net pay of $5,362.50",
        ],
        "expected_response": "EMP-001 January 2026: gross pay $8,333.33, net pay $5,362.50.",
    },
    {
        "scenario_id": "pto-planning-session",
        "turns": [
            "How many PTO days do I have left? My employee ID is EMP-001.",
            "Great. I'd like to take December 23 to December 25 off. Please submit a request.",
            "Remind me — what is the policy on rolling over unused PTO?",
        ],
        "expected_trajectory": ["get_pto_balance", "submit_pto_request", "lookup_hr_policy"],
        "assertions": [
            "Agent correctly reported 10 remaining PTO days for EMP-001 in turn 1",
            "Agent submitted a PTO request for December 23-25, 2026 in turn 2",
            "Agent correctly stated the 5-day PTO rollover limit in turn 3",
        ],
        "expected_response": None,
    },
    {
        "scenario_id": "new-employee-onboarding",
        "turns": [
            "I just joined the company. What is the remote work policy?",
            "How much PTO do I get as a new employee?",
            "What life insurance benefit does the company provide?",
            "Can you check the current PTO balance for employee EMP-042?",
        ],
        "expected_trajectory": [
            "lookup_hr_policy",
            "lookup_hr_policy",
            "get_benefits_summary",
            "get_pto_balance",
        ],
        "assertions": [
            "Agent looked up the remote work policy in turn 1",
            "Agent looked up the PTO policy in turn 2",
            "Agent described the life insurance benefit in turn 3",
            "Agent reported 13 remaining PTO days for EMP-042 in turn 4",
        ],
        "expected_response": None,
    },
]

print(f"Dataset has {len(DATASET)} scenarios.")

In [ ]:
# Invoke the agent for each scenario and collect the resulting session IDs.
dataset_session_records = []
for scenario in DATASET:
    sid = run_session(scenario["turns"], scenario["scenario_id"])
    dataset_session_records.append({"scenario": scenario, "session_id": sid})

print(f"\nCollected {len(dataset_session_records)} dataset sessions.")
print("Waiting 60s for CloudWatch span ingestion ...")
time.sleep(60)
print("Dataset sessions ready for evaluation.")

## Step 10: Batch Evaluation

Batch evaluation is the right tool when you want to evaluate many sessions at
once against the same set of evaluators and ground truth. Rather than looping
and calling `bac.evaluate(...)` per `(session, evaluator)` pair, we submit
all sessions in a single `StartBatchEvaluation` request and poll for
completion.

Workflow:

1. `bac.start_batch_evaluation(...)` — submit the batch with:
   - `name` (1-128 chars, unique per request)
   - `evaluationConfig.evaluators` — list of evaluator IDs to run
   - `sessionSource.cloudWatchSource` — service + log groups + session IDs
   - `sessionMetadata` — optional per-session ground truth
   - `clientToken` — idempotency token
2. `bac.get_batch_evaluation(batchEvaluateId=...)` — poll until status is
   one of `COMPLETED`, `FAILED`, or `STOPPED` (terminal states from the SDK
   `BatchEvaluateStatus` enum).
3. Read `evaluationResults.evaluatorSummaries[]` for per-evaluator
   `averageScore`, `totalEvaluated`, `totalFailed`.

In [ ]:
# Build sessionMetadata entries from the dataset records so the batch run
# can compare each session against its ground truth.
session_ids = [rec["session_id"] for rec in dataset_session_records]
session_metadata = []
for rec in dataset_session_records:
    scenario = rec["scenario"]
    ground_truth_inline = {
        "expectedTrajectory": {"toolNames": scenario["expected_trajectory"]},
        "assertions": [{"text": a} for a in scenario["assertions"]],
    }
    session_metadata.append({
        "sessionId": rec["session_id"],
        "testScenarioId": scenario["scenario_id"],
        "groundTruth": {"inline": ground_truth_inline},
    })

batch_name = f"gt-boto3-batch-{uuid.uuid4().hex[:8]}"
client_token = str(uuid.uuid4())

print(f"Submitting batch evaluation: {batch_name}")
start_resp = bac.start_batch_evaluation(
    name=batch_name,
    evaluationConfig={
        "evaluators": [
            {"evaluatorId": "Builtin.Correctness"},
            {"evaluatorId": "Builtin.GoalSuccessRate"},
            {"evaluatorId": "Builtin.TrajectoryExactOrderMatch"},
        ],
    },
    sessionSource={
        "cloudWatchSource": {
            "serviceNames": [SERVICE_NAME],
            "logGroupNames": ["aws/spans", CW_LOG_GROUP],
            "sessionInput": {"sessionIds": session_ids},
        },
    },
    sessionMetadata=session_metadata,
    clientToken=client_token,
)

batch_id = start_resp["batchEvaluateId"]
print(f"batchEvaluateId : {batch_id}")

In [ ]:
# Poll until the batch reaches a terminal state.
TERMINAL_STATUSES = {"COMPLETED", "FAILED", "STOPPED"}
POLL_SECONDS = 30

print(f"Polling get_batch_evaluation every {POLL_SECONDS}s ...")
while True:
    resp = bac.get_batch_evaluation(batchEvaluateId=batch_id)
    status = resp["status"]
    print(f"  status = {status}")
    if status in TERMINAL_STATUSES:
        break
    time.sleep(POLL_SECONDS)

batch_final = resp
print(f"\nBatch evaluation finished with status: {batch_final['status']}")

In [ ]:
# Surface evaluatorSummaries — per-evaluator averageScore / totalEvaluated / totalFailed.
results = batch_final.get("evaluationResults", {})
summaries = results.get("evaluatorSummaries", [])

if not summaries:
    display(Markdown("_No evaluator summaries returned. Check the batch status and CloudWatch log groups._"))
else:
    rows = [
        "| Evaluator | Name | Avg Score | Total Evaluated | Total Failed |",
        "|---|---|---|---|---|",
    ]
    for s in summaries:
        evaluator_id = s.get("evaluatorId", "")
        name = s.get("evaluatorName", "")
        stats = s.get("statistics", {})
        avg = stats.get("averageScore", "N/A")
        total_eval = s.get("totalEvaluated", "N/A")
        total_failed = s.get("totalFailed", "N/A")
        rows.append(
            f"| `{evaluator_id}` | {name} | {avg} | {total_eval} | {total_failed} |"
        )
    display(Markdown("### Batch Evaluation Summary\n\n" + "\n".join(rows)))

print("\nBatch evaluation complete.")